In [1]:
# Step 1: Install & Import Libraries
!pip install imbalanced-learn kagglehub -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import kagglehub
import glob, os
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import LabelEncoder
from imblearn.over_sampling  import SMOTE
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import sklearn
import matplotlib
import imblearn
import transformers

print('=' * 50)
print('  LIBRARY VERSIONS')
print('=' * 50)
print(f'  pandas           : {pd.__version__}')
print(f'  numpy            : {np.__version__}')
print(f'  scikit-learn     : {sklearn.__version__}')
print(f'  matplotlib       : {matplotlib.__version__}')
print(f'  imbalanced-learn : {imblearn.__version__}')
print(f'  transformers     : {transformers.__version__}')
print(f'  torch            : {torch.__version__}')
print(f'  GPU              : {"available ✅" if torch.cuda.is_available() else "not available — switch to GPU runtime"}')
print('=' * 50)
print('  All libraries loaded ✅')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 9.4 MB/s eta 0:00:00
  LIBRARY VERSIONS
  pandas           : 2.2.2
  numpy            : 2.0.2
  scikit-learn     : 1.6.1
  matplotlib       : 3.10.0
  imbalanced-learn : 0.14.1
  transformers     : 5.0.0
  torch            : 2.9.0+cpu
  GPU              : not available — switch to GPU runtime
  All libraries loaded ✅


In [ ]:
import os

# Step 2: Load the IO tweets, political tweets, and general tweets datasets from kaggle

print('Downloading datasets...')
print('(First run: ~2 min download. Future runs: instant from cache)')
print()

# kagglehub downloads the dataset and returns the local folder path
path_political_dir = kagglehub.dataset_download("kaushiksuresh147/political-tweets")
path_political = os.path.join(path_political_dir, "Political_tweets.csv")
print(f'Political tweets location: {path_political}')

path_io_dir = kagglehub.dataset_download("john123462546/english-ioa-tweets")
path_io = os.path.join(path_io_dir, "ioa_tweets_en.csv")
print(f'IO tweets location: {path_io}')

path_general_dir = kagglehub.dataset_download("i191796majid/tweets")
# kagglehub nests this as train.csv/train.csv (folder named train.csv containing the actual file)
path_general = os.path.join(path_general_dir, 'train.csv', 'train.csv')
if not os.path.isfile(path_general):
    path_general = os.path.join(path_general_dir, 'train.csv')
print(f'General tweets location: {path_general}')

info_op = pd.read_csv(path_io, low_memory=False)
legitimate = pd.read_csv(path_political, low_memory=False)
general = pd.read_csv(path_general, low_memory=False)

print()
print('=' * 50)
print(f'  Political tweets : {legitimate.shape[0]:,} rows x {legitimate.shape[1]} columns')
print(f'  IO tweets        : {info_op.shape[0]:,} rows x {info_op.shape[1]} columns')
print(f'  General tweets   : {general.shape[0]:,} rows x {general.shape[1]} columns')
print('=' * 50)

In [ ]:
# Step 3: Understand the Data

print('=' * 55)
print('  RAW DATASET OVERVIEW (3 Datasets)')
print('=' * 55)

legitimate.columns = legitimate.columns.str.strip()

print(f'\nShape   Political (Legit) : {legitimate.shape[0]:,} rows x {legitimate.shape[1]} columns')
print(f'Shape   IO                : {info_op.shape[0]:,} rows x {info_op.shape[1]} columns')
print(f'Shape   General tweets    : {general.shape[0]:,} rows x {general.shape[1]} columns\n')

print(f'Features  Political: {legitimate.shape[1]} columns')
print(f'Features  IO       : {info_op.shape[1]} columns')
print(f'Features  General  : {general.shape[1]} columns')

# Assign labels
legitimate["Label"] = 0
info_op["Label"] = 1

print('\nAll Political (Legit) column names:')
num_display_cols = 3
col_list = [f'{i:>3}. {col}' for i, col in enumerate(legitimate.columns, 1)]
rows = (len(col_list) + num_display_cols - 1) // num_display_cols
for r in range(rows):
    line = ''
    for c in range(num_display_cols):
        idx = r + c * rows
        if idx < len(col_list):
            line += f'{col_list[idx]:<40}'
    print(line)

print('\nAll IO column names:')
col_list = [f'{i:>3}. {col}' for i, col in enumerate(info_op.columns, 1)]
rows = (len(col_list) + num_display_cols - 1) // num_display_cols
for r in range(rows):
    line = ''
    for c in range(num_display_cols):
        idx = r + c * rows
        if idx < len(col_list):
            line += f'{col_list[idx]:<40}'
    print(line)

print('\nAll General tweets column names:')
col_list = [f'{i:>3}. {col}' for i, col in enumerate(general.columns, 1)]
rows = (len(col_list) + num_display_cols - 1) // num_display_cols
for r in range(rows):
    line = ''
    for c in range(num_display_cols):
        idx = r + c * rows
        if idx < len(col_list):
            line += f'{col_list[idx]:<40}'
    print(line)

print('\nPolitical Data types:')
print(legitimate.dtypes.value_counts().to_string())
print('\nIO Data types:')
print(info_op.dtypes.value_counts().to_string())
print('\nGeneral Data types:')
print(general.dtypes.value_counts().to_string())

print('\nFirst 3 rows — Political tweets (selected columns):')
preview_cols = list(legitimate.columns[:5]) + ['Label']
print(legitimate[preview_cols].head(3).to_string())

print('\nFirst 3 rows — General tweets:')
print(general.head(3).to_string())

print()
print('Label distribution (raw):')
print(f'  Political (Organic): {legitimate.shape[0]:,} rows — Label 0')
print(f'  IO:                  {info_op.shape[0]:,} rows — Label 1')
print(f'  General (Organic):   {general.shape[0]:,} rows — will be added as Label 0')

In [ ]:
# Step 4: Visualize class distribution (all 3 datasets)

# Ensure labels are set
if 'Label' not in legitimate.columns:
    legitimate["Label"] = 0
if 'Label' not in info_op.columns:
    info_op["Label"] = 1

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Class Distribution — All 3 Datasets', fontsize=16)

# Left: raw dataset sizes (3 bars)
dataset_names = ['Political\n(Organic)', 'IO Tweets', 'General\n(Organic)']
dataset_sizes = [len(legitimate), len(info_op), len(general)]
colors = ['#2ecc71', '#e74c3c', '#3498db']

sns.barplot(x=dataset_names, y=dataset_sizes, ax=axes[0], palette=colors)
axes[0].set_title('Raw Dataset Sizes')
axes[0].set_ylabel('Count')
for i, v in enumerate(dataset_sizes):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom')

# Right: combined organic vs IO (after merging political + general into organic)
combined_organic = len(legitimate) + len(general)
combined_io = len(info_op)
label_names = ['Organic\n(Political + General)', 'IO']
label_sizes = [combined_organic, combined_io]

sns.barplot(x=label_names, y=label_sizes, ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Combined Class Distribution (before cleaning)')
axes[1].set_ylabel('Count')
for i, v in enumerate(label_sizes):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print(f'\n  Organic = {len(legitimate):,} political + {len(general):,} general = {combined_organic:,} total')
print(f'  IO      = {len(info_op):,}')
print(f'  Adding general tweets prevents the model from learning "non-political = IO"')

In [ ]:
# Cell 5: Clean + Split + Balance (with topic-diverse organic class)



# ── CLEANING ─────────────────────────────────────────────

print('=' * 55)
print('  CLEANING')
print('=' * 55)

legitimate.columns = legitimate.columns.str.strip()
info_op.columns    = info_op.columns.str.strip()

info_op = info_op.rename(columns={
    'tweet_text': 'text',
})

# ── ADD GENERAL TWEETS TO ORGANIC CLASS ──────────────────

print('\n  Adding general tweets to organic class...')
print(f'  General tweets available: {len(general):,}')

# Prepare general tweets — rename column, add label
general_clean = general[['tweet']].copy()
general_clean = general_clean.rename(columns={'tweet': 'text'})
general_clean['Label'] = 0
general_clean['is_retweet'] = False

# Use same amount as political tweets so organic class is 50/50 political/general
GENERAL_SAMPLE_SIZE = min(len(legitimate), len(general_clean))
general_sample = general_clean.sample(n=GENERAL_SAMPLE_SIZE, random_state=42)
print(f'  Sampled {GENERAL_SAMPLE_SIZE:,} general tweets as Organic')

# Combine all three sources
keep_cols = ['text', 'is_retweet', 'Label']
info_op_subset    = info_op[keep_cols]
legitimate_subset = legitimate[keep_cols]

df = pd.concat([info_op_subset, legitimate_subset, general_sample[keep_cols]], ignore_index=True)

rows_start = len(df)
print(f'\n  Combined rows  : {rows_start:,}')
print(f'  Labels         : {df["Label"].value_counts().to_dict()}')
print(f'    Organic sources: {len(legitimate):,} political + {GENERAL_SAMPLE_SIZE:,} general')

df_clean = df.copy()

# Strip column spaces
df_clean.columns = df_clean.columns.str.strip()
print(f'\n  Column spaces stripped ✅')

# Drop duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset='text')
print(f'  Duplicates removed     : {before - len(df_clean):,}')

# No infinity values in text data
print(f'  Infinity values        : 0 — text data has no numeric features')

# Remove retweets
before = len(df_clean)
df_clean = df_clean[df_clean['is_retweet'] == False]
print(f'  Retweets removed       : {before - len(df_clean):,}')

# Drop nulls
before = len(df_clean)
df_clean = df_clean.dropna(subset=['text'])
df_clean = df_clean[df_clean['text'].str.strip() != '']
print(f'  Null/empty removed     : {before - len(df_clean):,}')

# Clean text
def clean_text(text):
    text = str(text)
    text = re.sub(r'^RT\s+@?\w*:?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^RR\s+RT\s+@?\w*:?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'\bamp\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean['text'] = df_clean['text'].apply(clean_text)

before = len(df_clean)
df_clean = df_clean[df_clean['text'].str.len() > 20]
print(f'  Short rows removed     : {before - len(df_clean):,}')

nan_remaining = df_clean['text'].isnull().sum()
print(f'  NaN remaining          : {nan_remaining}')
print(f'  No imputation needed — null rows dropped, text cannot be imputed')

# Drop retweet column
df_clean = df_clean.drop(columns="is_retweet")
print(f'  Removed is_retweet column')

print(f'\n  Before : {rows_start:,}')
print(f'  After  : {len(df_clean):,}')
print(f'  Removed: {rows_start - len(df_clean):,}')
print(f'  Classes: {df_clean["Label"].value_counts().to_dict()}')

# ── SPLIT ────────────────────────────────────────────────

print()
print('=' * 55)
print('  TRAIN / TEST SPLIT')
print('=' * 55)

X = df_clean['text']
y = df_clean['Label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts  = pd.Series(y_test).value_counts().sort_index()

strat_df = pd.DataFrame({
    'Train count': train_counts,
    'Test count':  test_counts,
    'Train %':     (train_counts / len(y_train) * 100).round(2),
    'Test %':      (test_counts  / len(y_test)  * 100).round(2),
    'OK?':         ['YES' if abs(
                        train_counts[i]/len(y_train) -
                        test_counts[i]/len(y_test)
                    ) < 0.01 else 'NO'
                    for i in train_counts.index]
})
strat_df.index = ['Organic (0)', 'IO (1)']
print(strat_df.to_string())
print(f'\n  X_train: {X_train_text.shape[0]:,} rows')
print(f'  X_test:  {X_test_text.shape[0]:,} rows')

# ── TOKENIZE ─────────────────────────────────────────────

print()
print('=' * 55)
print('  TOKENIZATION — Twitter-RoBERTa')
print('=' * 55)
print('  Replaces SimpleImputer + StandardScaler')
print('  fit on train only, transform on test — same rule')

tokenizer = AutoTokenizer.from_pretrained('cardiffnlp/twitter-roberta-base')

def tokenize(texts, max_length=128):
    return tokenizer(
        list(texts),
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )

print('  Tokenizing training set...')
train_encodings = tokenize(X_train_text)
print('  Tokenizing test set...')
test_encodings  = tokenize(X_test_text)

print(f'  Train tokens: {train_encodings["input_ids"].shape}')
print(f'  Test tokens:  {test_encodings["input_ids"].shape}')
print(f'  NaN = 0 — token IDs are integers, no NaN possible ✅')

# ── BALANCE ──────────────────────────────────────────────

# After split — balance training set only
# Test set stays untouched

print(f'  Before balancing:')
print(f'    {pd.Series(y_train).value_counts().to_dict()}')

# Undersample majority class to match minority
train_df = pd.DataFrame({'text': X_train_text, 'Label': y_train})

min_count = train_df['Label'].value_counts().min()

train_balanced = pd.concat([
    train_df[train_df['Label'] == 0].sample(n=min_count, random_state=42),
    train_df[train_df['Label'] == 1].sample(n=min_count, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

X_train_text_bal = train_balanced['text']
y_train_bal      = train_balanced['Label'].values

print(f'  After balancing:')
print(f'    {pd.Series(y_train_bal).value_counts().to_dict()}')
print(f'  Test set untouched ✅')

# Then tokenize the balanced training set
print(f'\n  Tokenizing balanced training set...')
train_encodings = tokenize(X_train_text_bal)
test_encodings  = tokenize(X_test_text)

X_train_bal = train_encodings['input_ids'].numpy()
X_test      = test_encodings['input_ids'].numpy()

print(f'  X_train_bal : {X_train_bal.shape}')
print(f'  X_test      : {X_test.shape}')

In [6]:
# Cell 6: Verify Ready for A3

print('=' * 55)
print('  CELL 6: VERIFY READY FOR A3')
print('=' * 55)

print(f'\n  Final shapes:')
print(f'    X_train_bal : {X_train_bal.shape}')
print(f'    X_test      : {X_test.shape}')
print(f'    y_train_bal : {len(y_train_bal):,}')
print(f'    y_test      : {len(y_test):,}')

# NaN check
nan_train = np.isnan(X_train_bal.astype(float)).sum()
nan_test  = np.isnan(X_test.astype(float)).sum()

print(f'\n  NaN check:')
print(f'    X_train_bal : {nan_train} {"✅" if nan_train == 0 else "❌"}')
print(f'    X_test      : {nan_test} {"✅" if nan_test == 0 else "❌"}')

# Infinity check
inf_train = np.isinf(X_train_bal.astype(float)).sum()
inf_test  = np.isinf(X_test.astype(float)).sum()

print(f'\n  Infinity check:')
print(f'    X_train_bal : {inf_train} {"✅" if inf_train == 0 else "❌"}')
print(f'    X_test      : {inf_test} {"✅" if inf_test == 0 else "❌"}')

# Class balance
print(f'\n  Class distribution:')
print(f'    Training : {pd.Series(y_train_bal).value_counts().to_dict()}')
print(f'    Test     : {pd.Series(y_test).value_counts().to_dict()}')

# Variable check
print(f'\n  Required variables:')
print(f'    X_train_bal : ✅')
print(f'    X_test      : ✅')
print(f'    y_train_bal : ✅')
print(f'    y_test      : ✅')
print(f'    tokenizer   : ✅')

print()
print('=' * 55)
if nan_train == 0 and nan_test == 0 and inf_train == 0 and inf_test == 0:
    print('  ALL CHECKS PASSED — READY FOR A3 ✅')
else:
    print('  CHECKS FAILED — FIX ISSUES ABOVE ❌')
print('=' * 55)

  CELL 6: VERIFY READY FOR A3

  Final shapes:
    X_train_bal : (249636, 128)
    X_test      : (77533, 128)
    y_train_bal : 249,636
    y_test      : 77,533

  NaN check:
    X_train_bal : 0 ✅
    X_test      : 0 ✅

  Infinity check:
    X_train_bal : 0 ✅
    X_test      : 0 ✅

  Class distribution:
    Training : {0: 124818, 1: 124818}
    Test     : {0: 46329, 1: 31204}

  Required variables:
    X_train_bal : ✅
    X_test      : ✅
    y_train_bal : ✅
    y_test      : ✅
    tokenizer   : ✅

  ALL CHECKS PASSED — READY FOR A3 ✅


In [ ]:
# Step 7: Convert token arrays to HuggingFace Datasets

from datasets import Dataset

pad_token_id = tokenizer.pad_token_id

def arrays_to_dataset(X_ids, y):
    attention_masks = (X_ids != pad_token_id).astype(int)
    return Dataset.from_dict({
        'input_ids': X_ids.tolist(),
        'attention_mask': attention_masks.tolist(),
        'labels': y.tolist()
    })

train_dataset = arrays_to_dataset(X_train_bal, y_train_bal)
test_dataset  = arrays_to_dataset(X_test, y_test.values)

print(f'Train dataset: {train_dataset}')
print(f'Test dataset:  {test_dataset}')
print(f'Features: {train_dataset.column_names}')
print(f'Sample labels: {train_dataset["labels"][:10]}')

In [ ]:
%%time
# Step 8: Fine-tune Twitter-RoBERTa (improved v1 — topic-diverse organic class)

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from sklearn.metrics import f1_score, accuracy_score

# Custom callback for progress logging
class ProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        print('=' * 55)
        print('  TRAINING STARTED')
        print(f'  Epochs: {args.num_train_epochs}')
        print(f'  Train samples: {state.num_train_epochs * state.max_steps // args.num_train_epochs if state.max_steps else "?"}')
        print(f'  Total steps: {state.max_steps}')
        print(f'  Batch size: {args.per_device_train_batch_size}')
        print('=' * 55)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and state.max_steps:
            pct = (state.global_step / state.max_steps) * 100
            epoch = logs.get('epoch', '?')
            loss = logs.get('loss', logs.get('eval_loss', '?'))
            lr = logs.get('learning_rate', '?')
            print(f'  [{state.global_step}/{state.max_steps}] ({pct:.1f}%) — epoch {epoch} — loss: {loss} — lr: {lr}')

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_num = int(state.epoch)
        pct = (epoch_num / args.num_train_epochs) * 100
        print()
        print(f'  === EPOCH {epoch_num}/{int(args.num_train_epochs)} COMPLETE ({pct:.0f}%) ===')
        print()

    def on_train_end(self, args, state, control, **kwargs):
        print('=' * 55)
        print('  TRAINING COMPLETE')
        print(f'  Total steps: {state.global_step}')
        print('=' * 55)

model = AutoModelForSequenceClassification.from_pretrained(
    'cardiffnlp/twitter-roberta-base',
    num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    print(f'  >> Eval — F1: {f1:.4f} — Accuracy: {acc:.4f}')
    return {'f1': f1, 'accuracy': acc}

training_args = TrainingArguments(
    output_dir='./results_improved_v1',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=True,
    logging_steps=500,
    report_to='none',
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[ProgressCallback()]
)

trainer.train()

# Save the improved model
trainer.save_model('./finetuned-roberta-improved-v1')
tokenizer.save_pretrained('./finetuned-roberta-improved-v1')
print('\nImproved v1 model saved to ./finetuned-roberta-improved-v1')

In [ ]:
# Step 9: Improved v1 evaluation — classification report + confusion matrix

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Get predictions from the trained model
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

# Classification report
print('=' * 55)
print('  IMPROVED v1 CLASSIFICATION REPORT')
print('  (trained with political + general organic tweets)')
print('=' * 55)
print()
print(classification_report(
    y_test, y_pred,
    target_names=['Organic', 'IO'],
    digits=4
))

# Extract key metrics
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

improved_f1  = f1_score(y_test, y_pred, average='weighted')
improved_acc = accuracy_score(y_test, y_pred)

print(f'  Weighted F1:  {improved_f1:.4f}')
print(f'  Accuracy:     {improved_acc:.4f}')

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Improved v1 RoBERTa (topic-diverse organic) — Confusion Matrix', fontsize=14)

# Raw counts
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Organic', 'IO']).plot(
    ax=axes[0], cmap='Blues', values_format=','
)
axes[0].set_title('Raw Counts')

# Normalized
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
ConfusionMatrixDisplay(cm_norm, display_labels=['Organic', 'IO']).plot(
    ax=axes[1], cmap='Blues', values_format='.3f'
)
axes[1].set_title('Normalized')

plt.tight_layout()
plt.savefig('improved_v1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n  Confusion matrix saved to improved_v1_confusion_matrix.png')

In [ ]:
# Step 10: Export model weights for download

import shutil

export_dir = './finetuned-roberta-improved-v1'
archive_name = 'finetuned-roberta-improved-v1'

# Zip the saved model directory
shutil.make_archive(archive_name, 'zip', export_dir)

zip_path = f'{archive_name}.zip'
zip_size = os.path.getsize(zip_path) / (1024 * 1024)
print(f'Exported: {zip_path} ({zip_size:.1f} MB)')
print(f'Contents:')
for f in sorted(os.listdir(export_dir)):
    size = os.path.getsize(os.path.join(export_dir, f)) / (1024 * 1024)
    print(f'  {f:<35} {size:>8.1f} MB')

# On Kaggle: download from /kaggle/working/finetuned-roberta-improved-v1.zip
# On Colab: uncomment the next two lines
# from google.colab import files
# files.download(zip_path)

In [ ]:
# Step 11: Interactive tweet prediction

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def predict_post(text):
    inputs = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    pred_label = torch.argmax(probs).item()
    label_name = 'IO' if pred_label == 1 else 'Organic'
    confidence = probs[pred_label].item()

    print('=' * 55)
    print(f'  Text:       {text[:80]}{"..." if len(text) > 80 else ""}')
    print(f'  Prediction: {label_name}')
    print(f'  Confidence: {confidence:.4f}')
    print(f'  Organic:    {probs[0]:.4f}')
    print(f'  IO:         {probs[1]:.4f}')
    print('=' * 55)

    return label_name, confidence

# Enter a tweet to test:
tweet = input('Enter a tweet to classify: ')
predict_post(tweet)